# Fine-tuning a model with the Trainer API

**What this notebook does in plain English:**

We take BERT (a model that already knows a lot about language from pre-training)
and *fine-tune* it — teach it a specific new task using labelled examples.

The task here is **MRPC** (paraphrase detection): given two sentences, predict
whether they mean the same thing (label 1) or not (label 0).

### Steps in this notebook:
1. Load + tokenize the MRPC dataset
2. Set training configuration (`TrainingArguments`)
3. Load BERT with a classification head (`AutoModelForSequenceClassification`)
4. Create a `Trainer` and call `.train()`
5. Evaluate predictions manually
6. Add automatic evaluation metrics so we see accuracy/F1 per epoch

> **Tip:** Run cells top-to-bottom. Each cell depends on the ones above it.

## Step 0 — Install libraries

Run this once. It installs:
- `transformers` — BERT, tokenizers, Trainer API
- `datasets` — fast dataset loading (MRPC lives here)
- `evaluate` — accuracy/F1 metrics
- `sentencepiece` — tokenizer backend used by some models

In [11]:
# Install everything needed — run this first before any other cell
!pip install datasets evaluate transformers[sentencepiece]

## Step 1 — Load the dataset and tokenize it

We need to:
1. Download the MRPC dataset (3,668 training sentence pairs)
2. Convert every sentence from words → token IDs (numbers BERT understands)
3. Set up dynamic padding so batches are memory-efficient

**Why tokenize?** BERT cannot read raw text. It reads integers from a fixed
vocabulary of ~30,000 words/subwords. The tokenizer does that conversion.

In [12]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

# ── 1a. Load MRPC (part of the GLUE benchmark) ───────────────────────────────
# Returns a DatasetDict with 3 splits: 'train', 'validation', 'test'
# Each example: {sentence1, sentence2, label (0/1), idx}
raw_datasets = load_dataset("glue", "mrpc")

# ── 1b. Load the tokenizer for BERT (lowercase, uncased vocabulary) ──────────
# 'bert-base-uncased' means: 12 transformer layers, ~110M params, lowercase input
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


# ── 1c. Define the tokenization function ────────────────────────────────────
# This function takes ONE example (or a batch) and converts it to token IDs.
# We pass BOTH sentences to the tokenizer together — BERT is designed for this.
# It produces: [CLS] sentence1 [SEP] sentence2 [SEP]
# truncation=True: cut any pair exceeding BERT's 512-token limit
# (No padding here — we pad dynamically per batch in the DataCollator below)
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)


# ── 1d. Apply tokenization to ALL splits at once ─────────────────────────────
# .map() runs tokenize_function over every example in train/validation/test.
# batched=True processes ~1000 examples at a time (much faster than one-by-one).
# New columns are added: input_ids, token_type_ids, attention_mask
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# ── 1e. Set up the DataCollator ──────────────────────────────────────────────
# Sequences in one batch may have different lengths.
# DataCollatorWithPadding pads them to the longest in THAT batch (not always 512).
# This saves memory vs padding everything to 512 all the time.
# It also sets attention_mask=0 on padding tokens so BERT ignores them.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

## Step 2 — Set training configuration (`TrainingArguments`)

`TrainingArguments` is just a big settings object — it tells the Trainer:
- Where to save checkpoints
- How many epochs to train
- What batch size and learning rate to use
- When (if ever) to evaluate

This first version uses all defaults — we'll improve it in Step 6.

In [13]:
from transformers import TrainingArguments

# TrainingArguments holds every hyperparameter for training.
# The only required argument is the output directory name.
# 'test-trainer' = folder where model checkpoints will be saved during training.
#
# Default values used here (you can override any of these):
#   learning_rate       = 5e-5
#   num_train_epochs    = 3
#   per_device_train_batch_size = 8
#   per_device_eval_batch_size  = 8
#   evaluation_strategy = 'no'  ← no evaluation during training (changed in Step 6)
#   weight_decay        = 0
training_args = TrainingArguments("test-trainer")

## Step 3 — Load BERT with a classification head

`AutoModelForSequenceClassification` takes the pre-trained BERT backbone
and adds a small **classification head** on top — a single linear layer
that maps BERT's output (768 numbers) to `num_labels` scores.

Training will update ALL weights: both the BERT backbone and the new head.

In [14]:
from transformers import AutoModelForSequenceClassification

# Load BERT with a classification head.
#
# from_pretrained() downloads the pre-trained BERT weights (~440 MB).
# num_labels=2 tells it we have 2 classes: 0=not_equivalent, 1=equivalent
# This adds a linear layer: BERT_output(768) → 2 scores (one per class)
#
# You'll see a WARNING: 'Some weights were not used... classifier.weight'
# This is EXPECTED — it means the new classification head has random weights
# (it was never pre-trained). Fine-tuning will train those weights.
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Step 4 — Create the Trainer

The `Trainer` is HuggingFace's high-level training loop. It handles:
- Batching data with the DataCollator
- Moving data to GPU automatically
- Running forward + backward pass
- Updating weights with AdamW optimizer
- Saving checkpoints
- Logging loss

You just plug in your model, args, datasets, and call `.train()`.

In [15]:
from transformers import Trainer

# Create the Trainer by plugging in all the pieces:
#
#   model             → the BERT model we loaded above (with classification head)
#   training_args     → our TrainingArguments config (epochs, lr, output dir, etc.)
#   train_dataset     → tokenized training examples (3,668 sentence pairs)
#   eval_dataset      → tokenized validation examples (408 pairs) for evaluation
#   data_collator     → pads each batch to the same length dynamically
#   processing_class  → the tokenizer (used internally for some data processing)
#
# NOTE: no compute_metrics here yet — we won't see accuracy/F1 after each epoch.
# We add that in Step 6 for the improved version.
trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

## Step 5 — Train!

`.train()` runs the full training loop:
- Loops over the training data for `num_train_epochs` epochs (default: 3)
- Each epoch = one full pass through all 3,668 training examples
- Prints a table showing loss, learning rate, and step count

> **On CPU:** This will take ~20-40 minutes. On GPU (Colab): ~3-5 minutes.
> Enable GPU in Colab: Runtime → Change runtime type → T4 GPU

In [16]:
# Start fine-tuning!
# The Trainer will print a progress table like:
#
#   Epoch | Training Loss | Step
#   ------|---------------|-----
#     1   |    0.5432     |  500
#     2   |    0.3211     | 1000
#     3   |    0.2145     | 1500
#
# Loss decreasing = model is learning!
# The trained model is saved to the 'test-trainer' folder when done.
trainer.train()

Step,Training Loss
500,0.497672
1000,0.262403


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1377, training_loss=0.3096840830063941, metrics={'train_runtime': 229.1495, 'train_samples_per_second': 48.021, 'train_steps_per_second': 6.009, 'total_flos': 405114969714960.0, 'train_loss': 0.3096840830063941, 'epoch': 3.0})

## Step 5b — Evaluate: get raw predictions

After training, we want to know how well the model performs.
The next 3 cells do this manually so you can see every step:
1. Get raw logits from the model
2. Convert logits → class labels
3. Compare to ground truth labels using the GLUE metric

In [17]:
# trainer.predict() runs the model on the validation set without updating weights.
# Returns a PredictionOutput namedtuple with:
#   .predictions  → raw model output (logits), shape: (408, 2)
#                   408 = number of validation examples
#                   2   = one score per class (not_equivalent, equivalent)
#   .label_ids    → true labels (0 or 1), shape: (408,)
#   .metrics      → loss + runtime stats
#
# Logits are RAW scores — not probabilities yet. Higher score = model prefers that class.
predictions = trainer.predict(tokenized_datasets["validation"])
print(predictions.predictions.shape, predictions.label_ids.shape)
# Expected output: (408, 2) (408,)

(408, 2) (408,)


In [18]:
import numpy as np

# Convert logits → predicted class labels.
# Each row in predictions.predictions has 2 scores: [score_class0, score_class1]
# np.argmax(..., axis=-1) picks the index of the highest score per row.
# So score_class1 > score_class0  →  predicted label = 1 (equivalent)
#    score_class0 > score_class1  →  predicted label = 0 (not equivalent)
#
# Result: preds is a 1D array of 408 predicted labels (0s and 1s)
preds = np.argmax(predictions.predictions, axis=-1)

In [19]:
import evaluate

# Load the official GLUE/MRPC metric.
# For MRPC this returns two scores:
#   'accuracy' → what % of examples were classified correctly
#   'f1'       → F1 score (balances precision and recall — better for imbalanced data)
#
# A fine-tuned BERT on MRPC typically gets ~84-86% accuracy and ~88-90% F1.
metric = evaluate.load("glue", "mrpc")
metric.compute(predictions=preds, references=predictions.label_ids)
# Example output: {'accuracy': 0.8529, 'f1': 0.8982}

{'accuracy': 0.8504901960784313, 'f1': 0.8942807625649913}

## Step 6 — Improved Trainer with automatic evaluation

The first trainer above only showed training loss — we couldn't see accuracy/F1
during training. Now we:
1. Define a `compute_metrics` function the Trainer calls automatically after each epoch
2. Set `evaluation_strategy='epoch'` so evaluation runs at the end of every epoch

This way we can watch accuracy improve epoch by epoch and stop early if needed.

In [20]:
def compute_metrics(eval_preds):
    # The Trainer calls this function after each evaluation pass.
    # eval_preds is a tuple: (logits, labels)
    #   logits → raw model scores, shape (num_examples, num_labels)
    #   labels → true class labels, shape (num_examples,)
    metric = evaluate.load("glue", "mrpc")
    logits, labels = eval_preds

    # Convert logits to predicted class indices (same as before)
    # argmax picks the column (class) with the highest score for each row
    predictions = np.argmax(logits, axis=-1)

    # Compute and return the MRPC metrics dict: {'accuracy': ..., 'f1': ...}
    # The Trainer will log these values automatically after every evaluation step
    return metric.compute(predictions=predictions, references=labels)

### Create the improved Trainer

Same as before but with two additions:
- `evaluation_strategy='epoch'` → evaluate on the validation set after every epoch
- `compute_metrics=compute_metrics` → print accuracy and F1 after each evaluation

In [22]:
from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification

In [25]:
# Re-create TrainingArguments with evaluation turned on.
# evaluation_strategy='epoch' means: at the end of each epoch,
# run the model on eval_dataset and call compute_metrics().
# You'll now see accuracy and F1 printed in the training table.

# Re-load a fresh copy of the model (random classification head weights again).
# We do this to start training from scratch — not from the already-trained model above.
# If you skip this, you'd be fine-tuning an already fine-tuned model.

# Build the improved Trainer:
#   compute_metrics → called after every epoch's evaluation
#                     returns {'accuracy': ..., 'f1': ...} which the Trainer logs
from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification

training_args = TrainingArguments(
    output_dir="test-trainer",
    eval_strategy="epoch"
)

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,   # ✅ updated for v5.x
    compute_metrics=compute_metrics,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [26]:
import transformers
print(transformers.__version__)

5.3.0


### Train the improved model

This time you'll see accuracy and F1 in the training table after each epoch.

Expected output (approximately):
```
Epoch | Training Loss | Validation Loss | Accuracy | F1
  1   |    0.546      |     0.388       |  0.833   | 0.882
  2   |    0.318      |     0.448       |  0.845   | 0.896
  3   |    0.191      |     0.535       |  0.853   | 0.898
```
> **Note:** Exact numbers vary slightly each run due to random initialisation.

In [27]:
# Train the model — same as before but now we get metrics after every epoch.
# Watch the Accuracy and F1 columns improve with each epoch.
# If validation loss starts INCREASING while training loss keeps decreasing,
# that's overfitting — the model is memorising training data instead of generalising.
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.396828,0.830882,0.886700
2,0.511616,0.637140,0.835784,0.885470
3,0.258466,0.817296,0.848039,0.894198


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1377, training_loss=0.30753456619916486, metrics={'train_runtime': 230.5241, 'train_samples_per_second': 47.735, 'train_steps_per_second': 5.973, 'total_flos': 405114969714960.0, 'train_loss': 0.30753456619916486, 'epoch': 3.0})

In [29]:
trainer.save_model("my_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [30]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="my_model"
)

result = classifier("This movie is very good")
print(result)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': 'LABEL_0', 'score': 0.9418827295303345}]


In [31]:
texts = [
    "I love this product",
    "This is very bad",
    "Average experience"
]

for text in texts:
    print(text, "->", classifier(text))

I love this product -> [{'label': 'LABEL_0', 'score': 0.7341068983078003}]
This is very bad -> [{'label': 'LABEL_0', 'score': 0.671184778213501}]
Average experience -> [{'label': 'LABEL_1', 'score': 0.6182791590690613}]


In [32]:
tokenizer.save_pretrained("my_model")

('my_model/tokenizer_config.json', 'my_model/tokenizer.json')